In [ ]:
import pandas as pd
import numpy as np
import json, re

# Torch, Sklearn imports
from sklearn.model_selection import train_test_split
import torch
import torch.optim as optim
import torch.nn as nn
import torch.nn.functional as F
from torch.autograd import Variable
from torch.utils.data import Dataset, DataLoader
print(torch.__version__)

## Embeddings
import allennlp
from allennlp.modules.elmo import Elmo, batch_to_ids
from gensim.models import KeyedVectors

## NLP libs
from nltk import download
import gensim
from nltk.corpus import stopwords
download('stopwords')

## Importing SNIPS intent dataset

In [ ]:
dataset = pd.DataFrame(columns = ['phrase', 'intent'])

In [ ]:
for intent in ['AddToPlaylist', 'BookRestaurant', 'GetWeather', 'PlayMusic', 'RateBook', 'SearchCreativeWork',
               'SearchScreeningEvent']:
    with open("./2017-06-custom-intent-engines/" + intent + "/train_" + intent + ".json",
              encoding='cp1251') as data_file:
        data = json.load(data_file)
    print("Intent: {}, Length: {}".format(intent,len(data[intent])))
    texts = []
    for i in range(len(data[intent])):
        text = ''
        for j in range(len(data[intent][i]['data'])):
            text += data[intent][i]['data'][j]['text']
        dataset = dataset.append({'phrase': text, 'intent': intent}, ignore_index=True)

In [ ]:
dataset.intent.unique()

In [ ]:
def transformText(text, do_stop=False, do_stem=False):
    stops = set(stopwords.words("english"))
    # Convert text to lower
    text = text.lower()
    
    # Cleaning input
    text = text.replace("'s","")
    text = text.replace("?s","")
    text = text.replace("?","")
    text = text.replace("-","")
    
    # Removing non ASCII chars    
    text = re.sub(r'[^\x00-\x7f]',r' ',text)
    # Strip multiple whitespaces
    text = gensim.corpora.textcorpus.strip_multiple_whitespaces(text)
    # Removing all the stopwords
    if (do_stop==True):
        filtered_words = [word for word in text.split() if word not in stops]
    else:
        filtered_words = [word for word in text.split()]
    # Preprocessed text after stop words removal
    text = " ".join(filtered_words)
    # Remove the punctuation
    text = gensim.parsing.preprocessing.strip_punctuation2(text)
    # Strip multiple whitespaces
    text = gensim.corpora.textcorpus.strip_multiple_whitespaces(text)
    if (do_stem==True):
        # Stemming
        text = gensim.parsing.preprocessing.stem_text(text)
    return text

def strip_punctuation(s):
    return ''.join(c for c in s if c not in PUNCT)

## Lemmatization function based on Spacy Library
def lemmatizer_spacy(text):        
    sent = []
    doc = spacy_en(text)
    for word in doc:
        if word.lemma_ == "-PRON-":
            sent.append(word.text)
        else:
            sent.append(word.lemma_)
    return " ".join(sent)

def strip_punctuation(s):
    return ''.join(c for c in s if c not in punctuation)

In [ ]:
dataset['preproc_text'] = dataset['phrase'].apply(lambda x: transformText(x))

In [ ]:
dataset.tail(10)

In [ ]:
!ls ../../../vectors/

## Build Vocabulary

In [ ]:
## Build word vocabulary
word_to_ix = {}
for sent in dataset.preproc_text:
    for word in sent.split():
        if word not in word_to_ix:
            word_to_ix[word] = len(word_to_ix)
print("Tamanho do dicionario: {}".format(len(word_to_ix)))

In [ ]:
## Build label vocabulary
label_to_ix = {}
for label in dataset.intent:
    for word in label.split():
        if word not in label_to_ix:
            label_to_ix[word]=len(label_to_ix)

In [ ]:
label_to_ix

## Preparing PyTorch Dataset Loader

In [ ]:
class Intents(Dataset):
    def __init__(self, dataframe, w2v_weights_path):
        self.len = len(dataframe)
        self.label_to_ix = {}
        self.data = dataframe
        self.w2v = w2v = KeyedVectors.load_word2vec_format(w2v_weights_path, binary = True)
        
    def __getitem__(self, index):
        phrase = self.data.preproc_text[index]
        X, _  = self.get_avg_sentence_vector(phrase)
        y = label_to_ix[self.data.intent[index]]
        #X.requires_grad = False
        #y.requires_grad = False
        #blin = X.detach()
        return X, y
    
    def __len__(self):
        return self.len

    def get_avg_sentence_vector(self, sentence):
        featureVec = np.zeros((self.w2v.vector_size), dtype="float32")
        nwords = 0
        not_found_words = []
        for word in sentence.split():
            if word in self.w2v.index2word:
                nwords = nwords+1
                featureVec = np.add(featureVec, self.w2v.get_vector(word))
            else:
                not_found_words.append(word)
        if nwords>0:
            featureVec = np.divide(featureVec, nwords)
        return featureVec, not_found_words

In [ ]:
# Set data locations for embeddings
elmo_config_key_path = '../../../vectors/elmo_2x4096_512_2048cnn_2xhighway_options.json'
elmo_weights_key_path = '../../../vectors/elmo_2x4096_512_2048cnn_2xhighway_weights.hdf5'
w2v_weights_path = '../../../vectors/GoogleNews-vectors-negative300.bin'

In [ ]:
train_size = 0.8
train_dataset=dataset.sample(frac=train_size,random_state=200).reset_index(drop=True)
test_dataset=dataset.drop(train_dataset.index).reset_index(drop=True)

In [ ]:
print("FULL Dataset: {}".format(dataset.shape))
print("TRAIN Dataset: {}".format(train_dataset.shape))
print("TEST Dataset: {}".format(test_dataset.shape))

In [ ]:
training_set = Intents(train_dataset,  w2v_weights_path)
testing_set = Intents(test_dataset, w2v_weights_path)

In [ ]:
testing_set.__getitem__(3)

## Simple MLP Classifier

In [ ]:
class SimpleMLP(nn.Module):
    def __init__(self, inputdim, 
                        nclasses, 
                        nhidden, 
                        dropout = 0,
                        cudaEfficient=True):
        super(SimpleMLP, self).__init__()
        """
        PARAMETERS:
        -dropout:    dropout for MLP
        """
        
        self.inputdim = inputdim
        self.hidden_dim = nhidden
        self.dropout = dropout
        self.nclasses = nclasses
        
        if cudaEfficient:
            self.model = nn.Sequential(
                nn.Linear(self.inputdim, nhidden),
                nn.Dropout(p=self.dropout),
                nn.ReLU(),
                nn.Linear(nhidden, self.nclasses),
                ).cuda()
        else:
            self.model = nn.Sequential(
                nn.Linear(self.inputdim, nhidden),
                nn.Dropout(p=self.dropout),
                nn.ReLU(),
                nn.Linear(nhidden, self.nclasses),
                )
    def forward(self, x):
        log_probs = self.model(x)
        return log_probs

In [ ]:
INP_DIM = training_set.w2v.vector_size
NUM_LABELS = len(label_to_ix)
NHIDDEN = 512
DROPOUT = 0.3
model = SimpleMLP(inputdim = INP_DIM ,
          nhidden = NHIDDEN,
          nclasses = NUM_LABELS,
          dropout = DROPOUT, 
          cudaEfficient = False)

In [ ]:
NUM_LABELS

## Training

In [ ]:
# Parameters
params = {'batch_size': 64,
          'shuffle': True,
          'num_workers': 1}

In [ ]:
training_loader = DataLoader(training_set, **params)
testing_loader = DataLoader(testing_set, **params)

In [ ]:
training_loader.batch_size

In [ ]:
loss_function = nn.CrossEntropyLoss()
learning_rate = 0.001
optimizer = optim.Adam(params =  model.parameters(), lr=learning_rate)

In [ ]:
max_epochs = 10
for epoch in range(max_epochs):
    print("EPOCH -- {}".format(epoch))
    for i, (sent, label) in enumerate(training_loader):
        ## Step 1 - Clear gradients w.r.t. parameters
        optimizer.zero_grad()
        
        sent = Variable(sent)
        label = Variable(label)
        
        ## Step 2 - Run forward pass
        output = model.forward(sent)
        
        # Get predictions from the maximum value
        _, predicted = torch.max(output.data, 1)
        
        ## Step 3 - Compute loss
        loss = loss_function(output, label)
        loss.backward()
        
        ## Step 4 = Update parameters
        optimizer.step()
        
        if i%100 == 0:
            
            # Calculate Accuracy         
            correct = 0
            total = 0
            
            for sent, label in testing_loader:
                sent = Variable(sent)
                label = Variable(label)
                
                # Forward pass only to get logits/output
                output = model.forward(sent)
                
                # Get predictions from the maximum value
                _, predicted = torch.max(output.data, 1)
                
                # Total number of labels
                total += label.size(0)

                # Total correct predictions
                correct += (predicted.cpu() == label.cpu()).sum()
            accuracy = 100.00 * correct.numpy() / total
            # Print Loss
            print('Iteration: {}. Loss: {}. Accuracy: {}%'.format(i, loss.data[0], accuracy))

In [ ]:
input_phrase = "i need to book a restaurant today"

In [ ]:
def get_reply(phrase):
    inp, _ = training_set.get_avg_sentence_vector(phrase)
    inp = Variable(torch.Tensor(inp))
    output = model.forward(inp)

    # Get predictions from the maximum value
    _, predicted = torch.max(output.data, 0)
    pred_label=list(label_to_ix.keys())[list(label_to_ix.values()).index(predicted.item())]
    return pred_label

In [ ]:
get_reply(input_phrase)

In [ ]:
label_to_ix

## Intent Classifier w/ Uncertainty - MC Dropout

In [ ]:
class MLPUncertainty(nn.Module):
    def __init__(self, inputdim, 
                        nclasses, 
                        nhidden, 
                        dropout = 0,
                        cudaEfficient=True,
                        decay = 1e-6):
        super(MLPUncertainty, self).__init__()
        """
        PARAMETERS:
        -dropout:    dropout for MLP
        """
        
        self.inputdim = inputdim
        self.hidden_dim = nhidden
        self.dropout = dropout
        self.decay = decay
        self.nclasses = nclasses
        
        if cudaEfficient:
            self.model = nn.Sequential(
                nn.Linear(self.inputdim, nhidden),
                nn.Dropout(p=self.dropout),
                nn.ReLU(),
                nn.Linear(nhidden, self.nclasses),
                ).cuda()
        else:
            self.model = nn.Sequential(
                nn.Linear(self.inputdim, nhidden),
                nn.Dropout(p=self.dropout),
                nn.ReLU(),
                nn.Linear(nhidden, self.nclasses),
                )
    def forward(self, x):
        log_probs = self.model(x)
        return log_probs

In [ ]:
def uncertainity_estimate(x, model, iters, l2):
    outputs = np.hstack([model(x).data.numpy() for i in trange(iters)])
    y_mean = outputs.mean(axis=1)
    y_variance = outputs.var(axis=1)
    tau = l2 * (1. - model.dropout_p) / (2. * N * model.decay)
    y_variance += (1. / tau)
    y_std = np.sqrt(y_variance)
    return y_mean, y_std

In [ ]:
INP_DIM = training_set.w2v.vector_size
NUM_LABELS = len(label_to_ix)
NHIDDEN = 512
DROPOUT = 0.3
model_uncertainty = MLPUncertainty(inputdim = INP_DIM ,
              nhidden = NHIDDEN,
              nclasses = NUM_LABELS,
              dropout = DROPOUT, 
              cudaEfficient = False)

## Training

In [ ]:
loss_function = nn.CrossEntropyLoss()
learning_rate = 0.001
optimizer = optim.Adam(params =  model_uncertainty.parameters(), lr=learning_rate)

In [ ]:
max_epochs = 10
for epoch in range(max_epochs):
    print("EPOCH -- {}".format(epoch))
    for i, (sent, label) in enumerate(training_loader):
        ## Step 1 - Clear gradients w.r.t. parameters
        optimizer.zero_grad()
        
        sent = Variable(sent)
        label = Variable(label)
        
        ## Step 2 - Run forward pass
        output = model_uncertainty.forward(sent)
        
        # Get predictions from the maximum value
        _, predicted = torch.max(output.data, 1)
        
        ## Step 3 - Compute loss
        loss = loss_function(output, label)
        loss.backward()
        
        ## Step 4 = Update parameters
        optimizer.step()
        
        if i%100 == 0:
            
            # Calculate Accuracy         
            correct = 0
            total = 0
            
            for sent, label in testing_loader:
                sent = Variable(sent)
                label = Variable(label)
                
                # Forward pass only to get logits/output
                output = model_uncertainty.forward(sent)
                
                # Get predictions from the maximum value
                _, predicted = torch.max(output.data, 1)
                
                # Total number of labels
                total += label.size(0)

                # Total correct predictions
                correct += (predicted.cpu() == label.cpu()).sum()
            accuracy = 100.00 * correct.numpy() / total
            # Print Loss
            print('Iteration: {}. Loss: {}. Accuracy: {}%'.format(i, loss.data[0], accuracy))

In [ ]:
x_test = 'play this music please'

In [ ]:
get_reply(x_test)

In [ ]:
inp, _ = training_set.get_avg_sentence_vector(x_test)
inp = Variable(torch.Tensor(inp))
output = model_uncertainty.forward(inp).data.numpy()
output

In [ ]:
iters_uncertainty = 50
lengthscale = 0.01
N = 1 

In [ ]:
def get_reply_uncertainty(phrase, model):
    inp, _ = training_set.get_avg_sentence_vector(phrase)
    inp = Variable(torch.Tensor(inp))
    logits_out = model.forward(inp)
    softmax_out = F.softmax(logits_out, dim=0)
    return softmax_out, logits_out

In [ ]:
def get_uncertainty(phrase, model, iters_uncertainty, lengthscale, N, softmax = True):
    if softmax:
        print("...Softmax Outputs")
        outputs = [get_reply_uncertainty(phrase, model)[0].data.numpy() for i in range(iters_uncertainty)]
    else:
        print("...Logit Outputs")
        outputs = [get_reply_uncertainty(phrase, model)[1].data.numpy() for i in range(iters_uncertainty)]
    
    outs = np.vstack(outputs)
    y_mean = outs.mean(axis=0)
    y_variance = outs.var(axis=0).sum()
    tau = lengthscale * (1. - model_uncertainty.dropout) / (2. * N * model_uncertainty.decay)
    y_variance += (1. / tau)
    y_std = np.sqrt(y_variance)
    pred_label = np.argmax(y_mean)
    pred_probs = np.max(y_mean)
    prediction=list(label_to_ix.keys())[list(label_to_ix.values()).index(pred_label)]
    return prediction, pred_probs, y_std

## Save and load model for demos

In [ ]:
model_file = "model__uncertainty.pth"
training_set_file = "training__1.pth"
testing_set_file = "testing__1.pth"

In [ ]:
torch.save(model_uncertainty, model_file)

In [ ]:
!ls

In [ ]:
modelito = torch.load("model__uncertainty.pth")

## Testing w/ Softmax Probabilities

In [ ]:
#phrase = 'trump is idiot'
x_test = ['Trump is an idiot',
          'Barack Obama is the coolest guy',
          'I will present uncertainty at TDC Porto Alegre', 
          'play blues music',
          'play Radiohead playlist',
          'give me the movie schedules at plitt theatres',
          'give me the movie schedules',
          'movie schedules',
          'schedules']
for utt in x_test:
    predicted, pred_probs, y_std = get_uncertainty(utt, modelito, iters_uncertainty, lengthscale, N, softmax = True)
    print("[INPUT]: {}".format(utt))
    print("[SOFTMAX PROBABILITIES]: {0:.2f}%".format(pred_probs*100))
    if y_std > 0.05:
        print(" --- UNCERTAIN --- ")
        print("[STD DEV]: {} ".format(y_std))
    else:
        print("[PREDICTED LABEL]: {}".format(predicted))
        print("[STD DEV]: {}".format(y_std))
    print("-"*50)

In [ ]:
train_dataset